# <MODEL NAME>: feature extractor evaluation

Copy this notebook to `NN_<model>.ipynb`, run it in the model's own Jupyter kernel (env in `envs/<model>/`), and fill in each section. Edit only the cells marked **TODO**.

There is no ground truth yet, so section 5 checks coverage, plausibility, and stability instead of accuracy.

In [ ]:
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path('..').resolve()

from src import clip_io, plots, schema

MODEL = '<model>'              # TODO: folder name used in outputs/
MODEL_DIR = ROOT / 'models' / '<MODEL_DIR>'  # TODO

## 1. Model card

Check every entry against the repo or paper. Do not claim an output is supported without a source.

| Field | Value |
|---|---|
| Paper | |
| Repo URL | |
| Commit hash | |
| License | |
| Inputs required | (face crop? size? normalization? BGR/RGB?) |
| Outputs | (which AUs, int and/or occ; gaze; head pose) |
| Native units / conventions | (AU range, gaze and pose units and sign directions) |
| Conversion to schema | (rescaling, rad to deg, sign flips; see `src/schema.py`) |
| Env | `envs/<model>/` (pyproject.toml + uv.lock), kernel `<model>` |

In [ ]:
!git -C {MODEL_DIR} remote get-url origin
!git -C {MODEL_DIR} rev-parse HEAD

## 2. Setup log

Record install steps, errors, and fixes in order, so the setup can be repeated on the lab machine.

| # | Step / command | Result | Error | Fix |
|---|---|---|---|---|
| 1 | | | | |

## 3. Run on test clips

`predict(image)` is the only model-specific code. It takes one BGR frame and returns `(face_conf, outputs)`: `outputs` is a dict of schema column names to values, already converted to schema units and signs, or `None` when no face is found. List every column it can return in `OUTPUT_COLUMNS`.

In [ ]:
# TODO: load the model
model = None

In [ ]:
# TODO: every column predict() can return, in a fixed order.
# Frames with no face get NaN in all of them, so clips with no detections still save.
OUTPUT_COLUMNS = []

def predict(image):
    """TODO: run the model on one BGR frame.

    Returns (face_conf, outputs). outputs is e.g.
    {'AU12_int': 1.3, 'gaze_yaw_deg': -4.2, ...} or None if no face.
    """
    raise NotImplementedError

In [ ]:
manifest = clip_io.load_manifest()
manifest

In [ ]:
results, timings = {}, {}
for row in manifest.itertuples():
    records = []
    t0 = time.perf_counter()
    for frame, timestamp_ms, image in clip_io.iter_frames(clip_io.clip_path(row)):
        face_conf, outputs = predict(image)
        record = {'clip_id': row.clip_id, 'frame': frame, 'timestamp_ms': timestamp_ms,
                  'face_conf': float(face_conf) if outputs is not None else 0.0}
        record.update(outputs or {})
        records.append(record)
    timings[row.clip_id] = time.perf_counter() - t0
    results[row.clip_id] = pd.DataFrame.from_records(records, columns=list(schema.REQUIRED_COLUMNS) + OUTPUT_COLUMNS)
    print(f'{row.clip_id}: {len(records)} frames in {timings[row.clip_id]:.1f} s')

## 4. Save output in the common schema

`save_output` runs `schema.validate` and refuses to write a file that fails it. Fix the conversion in `predict`, not the check.

In [ ]:
for clip_id, df in results.items():
    df = df.astype({'frame': 'int64', 'timestamp_ms': 'float64', 'face_conf': 'float64'})
    out_cols = [c for c in df.columns if c not in schema.REQUIRED_COLUMNS and not c.startswith(schema.EXTRA_PREFIX)]
    df[out_cols] = df[out_cols].astype('float64')
    df = schema.add_missing_au_pairs(df)
    print(clip_io.save_output(df, MODEL, clip_id))

## 5. Quick checks

These read back the saved files, so they test exactly what was written.

In [ ]:
out = clip_io.load_outputs(MODEL)
print('output columns:', schema.output_columns(out))

### Face-detection rate

In [ ]:
display(plots.detection_rate(out).to_frame())
plots.plot_detection_rate(out);

### Speed (fps on this machine)

Includes video decoding. Note the hardware (CPU/GPU) in the verdict.

In [ ]:
plots.fps_table(timings, out)

### Standard plots per clip

In [ ]:
for clip_id, df in out.groupby('clip_id'):
    plots.plot_face_conf(df)
    if schema.au_ids(df):
        for kind in ('int', 'occ'):
            if df.filter(like=f'_{kind}').notna().any().any():
                plots.plot_aus(df, kind)
    if any(c in df for c in schema.GAZE_COLUMNS):
        plots.plot_gaze(df)
    if any(c in df for c in schema.POSE_COLUMNS):
        plots.plot_pose(df)
    plt.show()

### Plausibility and stability (no ground truth)

- Value ranges: do angles and AU levels look reasonable for what happens in each clip?
- Jitter: median absolute frame-to-frame change on detected frames. A high value on a still face means a noisy model.
- AU occurrence rate: AUs that are always on or always off are suspicious.

In [ ]:
detected = out[out['face_conf'] >= plots.DEFAULT_CONF_THRESHOLD]
display(detected[schema.output_columns(out)].describe().T)

angle_cols = [c for c in schema.GAZE_COLUMNS + schema.POSE_COLUMNS if c in out]
if angle_cols:
    jitter = detected.groupby('clip_id')[angle_cols].apply(lambda d: d.diff().abs().median())
    display(jitter.rename(columns=lambda c: c.replace('_deg', '_jitter_deg')))

occ_cols = [c for c in out if c.endswith('_occ') and out[c].notna().any()]
if occ_cols:
    display(detected.groupby('clip_id')[occ_cols].mean())

### Failure cases

Frames where the face was missed or detected with low confidence. Look at a few to see why (pose, occlusion, lighting, blur).

In [ ]:
failures = plots.failure_frames(out)
display(failures.groupby('clip_id').size().rename('n_failures').to_frame())
failures.head(20)

In [ ]:
def show_frames(clip_id, frames, ncols=4):
    row = manifest.set_index('clip_id').loc[clip_id]
    wanted = set(frames)
    images = [(f, img) for f, _, img in clip_io.iter_frames(clip_io.DATA_DIR / row.video_path, rgb=True) if f in wanted]
    if not images:
        return
    nrows = -(-len(images) // ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(3 * ncols, 2.5 * nrows), squeeze=False)
    for ax in axes.flat:
        ax.axis('off')
    for ax, (f, img) in zip(axes.flat, images):
        ax.imshow(img)
        ax.set_title(f'{clip_id} frame {f}', fontsize='small')
    plt.show()

# Up to 8 evenly spaced failure frames per clip
for clip_id, f in failures.groupby('clip_id'):
    show_frames(clip_id, f['frame'].iloc[np.linspace(0, len(f) - 1, min(8, len(f))).astype(int)])

## 6. Notes and verdict

**Hardware used for the speed numbers:**

**Strengths:**

**Weaknesses:**

**Verdict:** keep / drop

**Why:**